# Build a supervisor multi-agent system

You will build a **router**. It reads an incoming content request, decides which of three
specialists should handle it — finance research, general research, or writing — and hands the
request to that specialist's own prompt and own tools.

Two model calls, four prompts, four tools, and **one trace** covering the whole thing.

A supervisor is not a framework or a state machine. It is one small model call whose only job is
to classify. That is worth saying plainly, because the pattern is usually presented as something
much heavier than it is.

| Piece | What it does | Who runs it |
|---|---|---|
| `content-supervisor` | the router prompt: describes the three specialists and asks for one | the platform stores it |
| `finance-research-agent` | subagent prompt, with three tools bound | the platform stores it |
| `general-research-agent` | subagent prompt, with two tools bound | the platform stores it |
| `writing-agent` | subagent prompt, with two tools bound | the platform stores it |
| `finance_research` | Yahoo Finance news for a ticker | **your code** |
| `basic_research` / `advanced_research` | Tavily search at two different depths | **your code** |
| `get_todays_date` | today's date, which a model cannot know | **your code** |

Every cell runs against a real account, a real model, and the real Yahoo Finance and Tavily APIs.
Nothing here is faked or mocked.

**This notebook needs two models on two different connection types.** Not for the routing — for
Step 12, which demonstrates a real trap: `response_format` behaves differently depending on how
the model's credential was connected. The preflight explains what to do if you only have one.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type, and **The same thing in code**, with a cell to run.
They are not two different features — the dashboard and the SDK call the same API, so the result
is identical. Pick either. Doing both is harmless, because every code cell looks for what already
exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a supervisor multi-agent system](https://docs.acruxcore.com/docs/tutorials/build-a-supervisor-multi-agent-system)

---

## Step 0 — What you need before you start

**1. A personal API key.** **Account & keys → New key**. Copy it the moment it appears — that is
the only time the full value is shown.

**2. A Tavily API key.** The free tier is enough. Get one at [tavily.com](https://tavily.com/).

**3. A model on a direct connection.** This matters more than usual here, and Step 1 says why.
A direct **OpenAI** or direct **Anthropic** connection handles `response_format` properly. This
notebook uses `claude-haiku` on a direct Anthropic connection.

**4. Optionally, a second model on an `openai_compatible` connection** — an OpenRouter credential,
for instance. Step 12 uses it to show what goes wrong. If you do not have one, that one cell will
say so and skip itself.

**5. `acruxcore` and `requests`.**

In [ ]:
%pip install -q --upgrade acruxcore requests

**Setup.** Set the keys and name everything this notebook will create.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("TAVILY_API_KEY", "tvly-...")

TAVILY_KEY = os.environ["TAVILY_API_KEY"]

MODEL = "claude-haiku"         # must be on a DIRECT openai or anthropic connection
TRACE_NAME = "content-supervisor-flow"   # both calls use this; Step 8 says why

ROUTER_PROMPT = "content-supervisor"

#: The router's answer is one of these keys; each maps to a stored prompt name.
SUBAGENT_PROMPTS = {
    "finance_research_agent": "finance-research-agent",
    "general_research_agent": "general-research-agent",
    "writing_agent": "writing-agent",
}

#: Which tools each subagent gets. The overlap is deliberate and comes from the source
#: this page ports: two subagents both use basic_research, and all three use the date tool.
TOOLS_BY_SUBAGENT = {
    "finance_research_agent": ["finance_research", "basic_research", "get_todays_date"],
    "general_research_agent": ["advanced_research", "get_todays_date"],
    "writing_agent": ["basic_research", "get_todays_date"],
}

ALL_TOOLS = sorted({name for names in TOOLS_BY_SUBAGENT.values() for name in names})

# Do NOT print the base URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Four things, in the order they fail. The third one is unusual and specific to this
notebook: it looks at *how each model's credential was connected*, because that changes whether
`response_format` works at all.

In [2]:
import httpx
import requests

from acruxcore import AcruxCore

hub = AcruxCore()          # reads ACRUXCORE_API_KEY / ACRUXCORE_BASE_URL

rest = httpx.AsyncClient(
    base_url=os.environ["ACRUXCORE_BASE_URL"],
    headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
    timeout=180,
)

# 1. Does the key work?
await hub.prompts.list(limit=1)
print("acruxcore key: ok")

# 2. and 3. Which models exist, and on what kind of connection?
models = (await rest.get("/gateway/models")).json()
print("\nmodels on this team:")
for model in models:
    print(f"  {model['publicName']:<16} provider={model['provider']:<18} "
          f"credential={model.get('credentialLabel')}")

DIRECT_PROVIDERS = {"openai", "anthropic"}
by_name = {m["publicName"]: m for m in models}
router_model = by_name.get(MODEL)

if router_model is None:
    print(f"\n!! MODEL {MODEL!r} is not on this team")
elif router_model["provider"] in DIRECT_PROVIDERS:
    print(f"\nMODEL {MODEL!r} is on a direct {router_model['provider']} connection - good, "
          "response_format is handled properly")
else:
    print(f"\n!! MODEL {MODEL!r} is on a {router_model['provider']} connection. "
          "Step 5's router may return prose instead of JSON - see Step 1.")

#: A model on an openai_compatible connection, if the team has one. Step 12 needs it.
PASSTHROUGH_MODEL = next(
    (m["publicName"] for m in models if m["provider"] not in DIRECT_PROVIDERS), None)
print("passthrough model for Step 12:", PASSTHROUGH_MODEL or "none - that cell will skip")

# 4. Does the Tavily key work?
probe = requests.post("https://api.tavily.com/search",
                      headers={"Authorization": f"Bearer {TAVILY_KEY}"},
                      json={"query": "test", "max_results": 1}, timeout=60)
print("tavily key:", "ok" if probe.ok else f"FAILED {probe.status_code}")

acruxcore key: ok

models on this team:
  mistral-small    provider=openai_compatible  credential=OpenRouter
  llama-3.3-70b    provider=openai_compatible  credential=OpenRouter
  claude-haiku     provider=anthropic          credential=Anthropic
  gemini-flash     provider=openai_compatible  credential=OpenRouter
  gpt-4o-mini      provider=openai             credential=OpenAI

MODEL 'claude-haiku' is on a direct anthropic connection - good, response_format is handled properly
passthrough model for Step 12: mistral-small
tavily key: ok


---

## Step 1 — What a supervisor actually is

### The general problem

One entry point, several specialists. A support desk with a billing team and a technical team. A
content pipeline with a research team and a copy team. Something has to read the incoming request
and decide who takes it.

The heavyweight answer is a state machine: a graph of nodes, a loop, and a text protocol the model
is asked to follow — reply `ROUTE_TO: x` to hand off, reply `COMPLETE` when finished. That works,
and it brings a framework, unbounded cycles, and string parsing with it.

### Where our case sits

The classification is just a model call. What made it need a protocol was that its answer was
free text. Constrain the answer instead and the protocol disappears:

```python
{"type": "object",
 "properties": {"route_to": {"type": "string", "enum": [...three names...]}},
 "required": ["route_to"], "additionalProperties": False}
```

Now the router cannot reply anything except one of three known strings. There is no text to parse,
so there is no parser to get wrong.

### The direct answer: two calls, one trace

| | Call A — classify | Call B — dispatch |
|---|---|---|
| prompt | `content-supervisor` | the subagent the router picked |
| `response_format` | the `route_to` schema | none |
| tools | **none** | that subagent's own tools |
| trace | opens the trace | **joins the same trace** |

The two calls use different prompts and different models are even possible, yet they are one run,
because Call B is passed the trace id Call A produced.

### Three traps, all real

**1. `response_format` and `tools` cannot both be set on one request.** That is why the table above
has them in different columns. Step 12 shows the error.

**2. The SDK cannot name a trace it opens with `chat()`.** It can give you the trace id — that is
`result.gateway.trace_id`, on the `gateway` object rather than on the result itself, which is easy
to miss. What it cannot do is name the trace. `hub.gateway.chat()` forwards only `tags` and
`metadata` from its `trace` argument, never a name, so the trace ends up auto-named after your
first user message. Worse, passing a `trace` dict on the gateway path makes the SDK report a second
`llm` span for a call the gateway already traced, so the span count doubles with no error.

Naming an opened trace therefore means sending the `x-trace-name` header yourself, which means one
raw `httpx` call. That is why Step 7 looks the way it does. It is not a workaround to apologise for
— naming the trace is what makes this flow findable later.

**3. `response_format` depends on the connection type.** The gateway handles it natively for a
direct **OpenAI** connection and translates it into a forced tool call for a direct **Anthropic**
one. For an `openai_compatible` connection — OpenRouter and friends — it is passed upstream
unchanged, and whether the upstream honours it is out of the gateway's hands. A router on such a
connection can return prose, and then `json.loads` fails. Step 12 demonstrates this against a real
`openai_compatible` model.

### The recommendation

Put the router on a direct connection, give it an enum, and thread the trace id. Reach for a graph
framework when you genuinely need multi-hop with cycles — this notebook builds the one-hop case,
which covers most routers.

---

## Step 2 — Write the four tool implementations

There is nothing to do in the dashboard for this step. These are functions in your process.

**Your app.** Two things worth noticing.

`basic_research` and `advanced_research` call the **same** Tavily endpoint with different
parameters. That is the whole difference between them: 5 results at basic depth versus 10 at
advanced. `basic_research` also prefixes the query with `trending`, which is what the source this
page ports does.

`finance_research` makes two HTTP calls in order — a `GET` for a session cookie, then the news
call that needs it. This notebook calls both endpoints directly with `requests` rather than
through `langchain_community`'s wrappers: fewer dependencies, and that package is being sunset.

In [3]:
from datetime import datetime

TAVILY_SEARCH = "https://api.tavily.com/search"
YAHOO_NEWS = "https://finance.yahoo.com/xhr/ncp?queryRef=latestNews&serviceKey=ncp_fin"


def _tavily(query: str, *, depth: str, max_results: int, images: bool) -> list:
    """One Tavily search. Both research tools are this function with different arguments."""
    res = requests.post(TAVILY_SEARCH, headers={"Authorization": f"Bearer {TAVILY_KEY}"},
                        json={"query": query, "search_depth": depth,
                              "max_results": max_results, "include_images": images},
                        timeout=90)
    res.raise_for_status()
    return [
        {"title": r["title"], "url": r["url"], "content": r["content"][:300]}
        for r in res.json().get("results", [])
    ]


async def finance_research(ticker_symbol: str) -> list:
    """Search Yahoo Finance news for a ticker symbol.

    Args:
        ticker_symbol: Stock ticker symbol, e.g. "TSLA".
    """
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.get("https://fc.yahoo.com", timeout=30)        # sets the session cookie
    res = session.post(YAHOO_NEWS,
                       json={"serviceConfig": {"snippetCount": 5, "s": [ticker_symbol]}},
                       timeout=60)
    res.raise_for_status()
    stream = res.json()["data"]["tickerStream"]["stream"]
    return [{"title": item["content"]["title"],
             "summary": (item["content"].get("summary") or "")[:300]} for item in stream[:5]]


async def advanced_research(query: str) -> list:
    """Deep web research: 10 results at advanced search depth.

    Args:
        query: What to research.
    """
    return _tavily(query, depth="advanced", max_results=10, images=False)


async def basic_research(query: str) -> list:
    """Quick web research: 5 results at basic search depth, for trending context.

    Args:
        query: What to research.
    """
    return _tavily(f"trending {query}", depth="basic", max_results=5, images=True)


async def get_todays_date() -> str:
    """Get today's date."""
    return datetime.now().strftime("%Y-%m-%d")


IMPLEMENTATIONS = {
    "finance_research": finance_research,
    "advanced_research": advanced_research,
    "basic_research": basic_research,
    "get_todays_date": get_todays_date,
}

print("today:", await get_todays_date())
print("finance_research(TSLA):", (await finance_research("TSLA"))[0]["title"][:80])
print("basic_research  hits:", len(await basic_research("AI content marketing")))
print("advanced_research hits:", len(await advanced_research("supervisor agent pattern")))

today: 2026-08-22
finance_research(TSLA): Elon Musk Is No Longer a Trillionaire, but He Could Still Give Every Person on E
basic_research  hits: 5
advanced_research hits: 10


---

## Step 3 — Commit the four tools to the catalog

The `@acrux.tool` decorator derives each tool's name, parameter schema and description from the
function — signature and docstring — and `hub.tools.sync()` reconciles the result with the
catalog. All four are `client` executors, because your process runs them.

### In the dashboard

**Gateway → Tools → New tool**, four times, then **New version** on each.

| Tool | Parameter | Description to type |
|---|---|---|
| `finance_research` | `ticker_symbol`, string, required | `Search Yahoo Finance news for a ticker symbol.` |
| `advanced_research` | `query`, string, required | `Deep web research: 10 results at advanced search depth.` |
| `basic_research` | `query`, string, required | `Quick web research: 5 results at basic search depth, for trending context.` |
| `get_todays_date` | none — add no rows | `Get today's date.` |

Set every **Executor** to **Client — the caller's app runs it**.

![The tool versions list showing the four tools committed](https://docs.acruxcore.com/img/tutorials/build-a-supervisor-multi-agent-system/02-tool-versions.png)

![The tools list with all four present](https://docs.acruxcore.com/img/tutorials/build-a-supervisor-multi-agent-system/03-tools-list.png)

### The same thing in code

**Setup.** `sync` is idempotent and cached per process on each spec's hash, so a second run of
this cell commits nothing and reports `committed=False`.

In [4]:
from acruxcore import acrux

DECORATED = {name: acrux.tool(fn) for name, fn in IMPLEMENTATIONS.items()}

results = await hub.tools.sync(list(DECORATED.values()))
TOOL_IDS = {}
for (name, fn), result in zip(DECORATED.items(), results):
    TOOL_IDS[name] = result.tool_id
    print(f"{name:>18}: v{result.version_number}  committed={result.committed}  "
          f"alias={result.alias}")

  finance_research: v2  committed=True  alias=production
 advanced_research: v1  committed=False  alias=production
    basic_research: v1  committed=False  alias=production
   get_todays_date: v1  committed=False  alias=production


**Check.** What the model will read for each one. `get_todays_date` is the interesting row: an
argument-free tool still needs a schema, and it is an object with no properties.

In [5]:
resolved = await hub.tools.resolve([{"name": name, "alias": "production"} for name in ALL_TOOLS])
for entry in resolved:
    params = list((entry.function.get("parameters") or {}).get("properties", {}))
    print(f"{entry.function['name']:>18}  v{entry.version_number}  {entry.executor_type}  "
          f"params={params or '(none)'}")

 advanced_research  v1  client  params=['query']
    basic_research  v1  client  params=['query']
  finance_research  v2  client  params=['ticker_symbol']
   get_todays_date  v1  client  params=(none)


---

## Step 4 — Create the router prompt

The router's system message does one job: describe the three specialists well enough that a model
can pick between them. Every word in those three descriptions is a routing decision waiting to
happen, so this is the prompt worth iterating on when routing goes wrong.

Note what the message does **not** contain: any instruction about output format. That comes from
`response_format` at call time, not from the prompt.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `content-supervisor` |
| **Description** | `Classifies an incoming content request and routes it to one of three specialist subagents.` |
| **Default model** | `claude-haiku`, on a direct connection |
| **System message** | the `ROUTER_SYSTEM` string in the next code cell |
| **User message** | `{{ question }}` |

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state, and
"the name exists" is not "it has content".

In [6]:
ROUTER_SYSTEM = """You are the Executive Content Director orchestrating a team of specialized
AI agents to produce exceptional content for clients.

Available agents:
- finance_research_agent: Specialized in financial data research and analysis using Yahoo
  Finance and other financial sources
- general_research_agent: Expert at comprehensive web research on any topic using advanced
  search tools
- writing_agent: Professional content writer that creates final polished content in any format

Read the user's request and decide which single agent should handle it next."""


async def find_prompt_by_name(name: str):
    """The prompt with exactly this name, or None. A notebook helper, NOT an SDK function."""
    found = await hub.prompts.list(search=name, limit=100)
    return next((p for p in found.data if p.name == name), None)


async def create_prompt_if_missing(name: str, *, description: str, system: str,
                                   variable: str) -> str:
    """Create the shell, then commit version 1 - but only where they are missing.

    A notebook helper, NOT an SDK function. It wraps the same two real calls used by hand
    for the router, hub.prompts.create() and hub.prompts.commit_version(), and skips
    whichever already exists so this notebook can be re-run. Returns the prompt id.
    """
    prompt = await find_prompt_by_name(name)
    if prompt is None:
        prompt = await hub.prompts.create(name=name, description=description)
        print(f"  + created shell {name}")
    else:
        print(f"  = shell {name} already exists")

    if (await hub.prompts.list_versions(prompt.id, limit=1)).total == 0:
        version = await hub.prompts.commit_version(
            prompt.id,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": "{{ " + variable + " }}"}],
            model=MODEL,
        )
        print(f"  + committed {name} v{version.version_number} on {version.model}")
    else:
        print(f"  = {name} already has a version")
    return prompt.id


router_id = await create_prompt_if_missing(
    ROUTER_PROMPT,
    description="Classifies an incoming content request and routes it to one of three "
                "specialist subagents.",
    system=ROUTER_SYSTEM,
    variable="question",
)
print("\nrouter prompt id:", router_id)

  + created shell content-supervisor
  + committed content-supervisor v1 on claude-haiku

router prompt id: 2fb775c3-dc24-42f3-8885-76a0be7742c8


---

## Step 5 — Create the three subagent prompts

Three prompts, one per specialist, each with its own system message and its own `{{ task }}`
variable.

Their tool lists overlap — `finance_research_agent` and `writing_agent` both use
`basic_research`, and all three use `get_todays_date` — but that does not make them the same
agent. Overlapping tools with different instructions is the normal case, not a smell.

### In the dashboard

**Prompts → New prompt**, three times, each with an **Editor** tab commit.

| Field | Value |
|---|---|
| **Names** | `finance-research-agent`, `general-research-agent`, `writing-agent` |
| **Default model** | `claude-haiku` for each |
| **System messages** | the three strings in the next code cell |
| **User message** | `{{ task }}` for each |

### The same thing in code

**Setup.** The same find-or-create helper from Step 4, called three more times.

In [7]:
SUBAGENT_SYSTEMS = {
    "finance-research-agent": (
        "You are an expert finance research assistant for a digital content agency.\n"
        "You have access to the following tools: finance_research, basic_research, and "
        "get_todays_date.\n"
        "First get today's date then continue.\n"
        "The finance_research tool is used to search for financial data and news from Yahoo "
        "Finance.\n"
        "The basic_research tool is used to search for general information.\n"
        "The get_todays_date tool is used to get today's date.\n"
        "When you are done with your research, return the research to the supervisor agent."
    ),
    "general-research-agent": (
        "You are an expert general research assistant for a digital content agency.\n"
        "You have access to the following tools: advanced_research and get_todays_date.\n"
        "First get today's date then continue.\n"
        "Use advanced_research for thorough, multi-source web research on any topic.\n"
        "Synthesise what you find across sources rather than summarising one of them.\n"
        "When you are done with your research, return the research to the supervisor agent."
    ),
    "writing-agent": (
        "You are a professional content writer for a digital content agency.\n"
        "You have access to the following tools: basic_research and get_todays_date.\n"
        "First get today's date then continue.\n"
        "Use basic_research to check what is currently trending on the topic before you "
        "write.\n"
        "Produce finished, polished copy in whatever format the request asks for - not notes "
        "and not an outline."
    ),
}

DESCRIPTIONS = {
    "finance-research-agent": "Finance research subagent for the content supervisor.",
    "general-research-agent": "General web research subagent for the content supervisor.",
    "writing-agent": "Writing subagent for the content supervisor.",
}

PROMPT_IDS = {ROUTER_PROMPT: router_id}
for prompt_name, system in SUBAGENT_SYSTEMS.items():
    PROMPT_IDS[prompt_name] = await create_prompt_if_missing(
        prompt_name,
        description=DESCRIPTIONS[prompt_name],
        system=system,
        variable="task",
    )

print("\nprompts on this team:", sorted(PROMPT_IDS))

  + created shell finance-research-agent
  + committed finance-research-agent v1 on claude-haiku
  + created shell general-research-agent
  + committed general-research-agent v1 on claude-haiku
  + created shell writing-agent
  + committed writing-agent v1 on claude-haiku

prompts on this team: ['content-supervisor', 'finance-research-agent', 'general-research-agent', 'writing-agent']


---

## Step 6 — Bind each subagent's own tools

This is where the three subagents stop being three copies of each other. Each gets only the tools
its instructions mention.

The router gets **no** tools at all. It is not allowed any: its call sets `response_format`, and
`response_format` and `tools` are mutually exclusive on one request.

### In the dashboard

**Prompts → each subagent → Tools tab → + Connect a tool from the catalog.**

| Prompt | Tools to connect |
|---|---|
| `finance-research-agent` | `finance_research`, `basic_research`, `get_todays_date` |
| `general-research-agent` | `advanced_research`, `get_todays_date` |
| `writing-agent` | `basic_research`, `get_todays_date` |
| `content-supervisor` | none — leave it empty |

Set each binding's alias to `production`, in the **default** column.

### The same thing in code

**Setup.** `set_tool_binding` replaces the binding for a tool rather than adding a second one, so
running this twice leaves the same seven rows.

In [8]:
for route, tool_names in TOOLS_BY_SUBAGENT.items():
    prompt_name = SUBAGENT_PROMPTS[route]
    for tool_name in tool_names:
        await hub.prompts.set_tool_binding(
            PROMPT_IDS[prompt_name], TOOL_IDS[tool_name], tool_alias="production")
    bindings = await hub.prompts.list_tool_bindings(PROMPT_IDS[prompt_name])
    print(f"{prompt_name:>24}: {sorted(b.tool_name for b in bindings.default)}")

router_bindings = await hub.prompts.list_tool_bindings(PROMPT_IDS[ROUTER_PROMPT])
print(f"{ROUTER_PROMPT:>24}: {[b.tool_name for b in router_bindings.default] or '(none, on purpose)'}")

  finance-research-agent: ['basic_research', 'finance_research', 'get_todays_date']
  general-research-agent: ['advanced_research', 'get_todays_date']
           writing-agent: ['basic_research', 'get_todays_date']
      content-supervisor: (none, on purpose)


---

## Step 7 — Call A: classify

**Your app.** The router call, and the one place this notebook drops out of the SDK.

`hub.gateway.chat()` would do the completion, and `result.gateway.trace_id` would even hand you the
id Call B needs. What it will not do is **name** the trace: `chat()` forwards only `tags` and
`metadata`, so `trace={"name": ...}` is ignored there and the trace gets auto-named after your
first user message. Passing that dict also makes the SDK report a duplicate `llm` span.

`x-trace-name` is the only thing that names a trace at the moment it opens, and a header needs a
raw request. So this one call is `httpx` — on purpose, for the name, not because the id is out of
reach. Everything after it joins by id.

In [9]:
ROUTE_SCHEMA = {
    "type": "object",
    "properties": {"route_to": {"type": "string", "enum": list(SUBAGENT_PROMPTS)}},
    "required": ["route_to"],
    "additionalProperties": False,
}

ROUTE_FORMAT = {
    "type": "json_schema",
    "json_schema": {"name": "route_decision", "schema": ROUTE_SCHEMA, "strict": True},
}


async def classify(question: str) -> tuple[str, str]:
    """Ask the router which subagent takes this. Returns (route_to, trace_id)."""
    rendered = await hub.prompts.render(ROUTER_PROMPT, "production", {"question": question})

    res = await rest.post(
        "/gateway/chat/completions",
        headers={"x-trace-name": TRACE_NAME},           # opens the trace and names it
        json={
            "model": rendered.model,
            "messages": rendered.messages,
            "response_format": ROUTE_FORMAT,
            "prompt_version_id": rendered.version_id,   # stamp WHICH prompt routed
            # No "tools" key: response_format and tools cannot both be set. See Step 10.
        },
    )
    res.raise_for_status()
    body = res.json()

    decision = json.loads(body["choices"][0]["message"]["content"])
    # The trace id is ONLY in this header - no SDK result type carries it.
    return decision["route_to"], res.headers["x-gateway-trace-id"]


route, trace_id = await classify(
    "Research Tesla (TSLA) latest stock news and tell me if investors should be worried.")
print("routed to:", route)
print("trace:    ", trace_id)

routed to: finance_research_agent
trace:     7f37e6fd-7115-47c4-a2ca-1aac14f8d615


---

## Step 8 — Call B: dispatch, on the same trace

**Your app.** Render the prompt the router picked, hand the subagent its own tools, and pass the
trace id from Call A so this lands in the same trace rather than a new one.

`sync=False` matters. The tools were committed in Step 3 and their definitions live in the
catalog; without this flag the loop would write the decorator's derived schema back over them on
every run.

`prompt_version_id` stamps the subagent's version onto the spans, which is what makes the trace
say *which* prompt did the work. Call A stamps the router's version the same way, so one trace ends
up carrying two prompt versions — that is the supervisor pattern visible in the data.

**The trap: `name` is repeated on purpose.** `run_tool_loop` labels its trace `runToolLoop` unless
you give it a name, and the gateway applies whatever name arrives last. Join a trace without
passing a name and the run works perfectly while the trace you named `content-supervisor-flow` in
Call A quietly becomes `runToolLoop`. Nothing errors; you just cannot find it by name later.

In [10]:
async def dispatch(route: str, task: str, trace_id: str) -> dict:
    """Run the chosen subagent's own prompt and tools, on an existing trace."""
    prompt_name = SUBAGENT_PROMPTS[route]
    rendered = await hub.prompts.render(prompt_name, "production", {"task": task})

    tools = [DECORATED[name] for name in TOOLS_BY_SUBAGENT[route]]
    result = await hub.gateway.run_tool_loop(
        rendered.model,
        rendered.messages,
        tools=tools,
        sync=False,                              # the catalog already owns these definitions
        prompt_version_id=rendered.version_id,   # stamp WHICH prompt did this work
        # Join Call A's trace. The name is repeated on purpose - see the lead-in above.
        trace={"trace_id": trace_id, "name": TRACE_NAME},
    )
    return {"prompt": prompt_name, "content": result.content,
            "turns": result.iterations, "trace_id": result.trace_id}


async def run_supervisor(question: str) -> dict:
    """The whole flow: classify, then dispatch to whoever was chosen."""
    route, trace_id = await classify(question)
    print(f"  routed to {route}  (trace {trace_id})")
    outcome = await dispatch(route, question, trace_id)
    outcome["route"] = route
    return outcome


QUESTIONS = [
    "Research Tesla (TSLA) latest stock news and tell me if investors should be worried.",
    "What are the biggest trends in sustainable packaging for consumer goods in 2026?",
    "Write a short LinkedIn post announcing our new AI research assistant.",
]

RUNS = []
for question in QUESTIONS:
    print(f"\nQ: {question}")
    outcome = await run_supervisor(question)
    RUNS.append(outcome)
    print(f"  {outcome['route']} answered in {outcome['turns']} turns:")
    print(f"  {outcome['content'][:260]}...")


Q: Research Tesla (TSLA) latest stock news and tell me if investors should be worried.
  routed to finance_research_agent  (trace a2f7ae23-149c-4f6f-8747-bf820dda7951)
  finance_research_agent answered in 2 turns:
  Based on my research of Tesla's latest stock news (as of August 22, 2026), here's my analysis:

## Key Findings:

**Positive Developments:**
- **Market Leadership:** Tesla controls 59% of the U.S. EV market — its highest share since 2023, demonstrating strong ...

Q: What are the biggest trends in sustainable packaging for consumer goods in 2026?
  routed to general_research_agent  (trace a79b1dea-7805-4439-aeb6-e65daf61b8b0)
  general_research_agent answered in 3 turns:
  Perfect! I now have comprehensive research data. Let me compile this into a thorough synthesis for you.

---

## **The Biggest Trends in Sustainable Packaging for Consumer Goods in 2026**

Based on current industry research, here are the major trends shaping s...

Q: Write a short LinkedIn post announcin

Three questions, three different subagents — and nothing in the code chose any of them. If the
routing were hardcoded, all three would have landed on the same prompt.

That is the test worth keeping. A router that always picks the same specialist looks exactly like
a working router until you send it a second kind of question.

---

## Step 9 — Read the traces back

**Check.** Each run is one trace holding two model calls that used **two different prompts**. That
is the thing to verify, and `prompt_version_id` on the spans is what makes it verifiable.

Two mechanics, both of which cost time when forgotten:

- **Flush first.** Spans are reported in the background, so a trace read immediately after a run
  can be incomplete.
- **Spans are a tree.** A tool span is a child of the model turn that asked for it, so counting
  them means walking `.children`.

![The trace overview showing the router call and the subagent's work in one trace](https://docs.acruxcore.com/img/tutorials/build-a-supervisor-multi-agent-system/05-trace-overview.png)

![The router span, with its structured route decision](https://docs.acruxcore.com/img/tutorials/build-a-supervisor-multi-agent-system/04-trace-router-span.png)

![A tool span inside the subagent's loop](https://docs.acruxcore.com/img/tutorials/build-a-supervisor-multi-agent-system/06-trace-tool-span.png)

In [11]:
await hub.gateway.flush()          # drain the background span queue before reading


def every_span(spans):
    """Flatten the span tree. A tool span is a child of the model turn that asked for it."""
    for span in spans:
        yield span
        yield from every_span(span.children)


for outcome in RUNS:
    detail = await hub.traces.get(outcome["trace_id"])
    spans = list(every_span(detail.spans))
    versions = sorted({s.prompt_version_id for s in spans if s.prompt_version_id})
    print(f"{outcome['route']}:")
    print(f"  trace {detail.trace.name}  spans={detail.trace.span_count}  "
          f"tokens={detail.trace.total_tokens}")
    print(f"  llm calls: {sum(1 for s in spans if s.kind == 'llm')}   "
          f"tools: {[s.name for s in spans if s.kind == 'tool']}")
    print(f"  distinct prompt versions stamped: {len(versions)}")

finance_research_agent:
  trace content-supervisor-flow  spans=5  tokens=3528
  llm calls: 3   tools: ['get_todays_date', 'finance_research']
  distinct prompt versions stamped: 2
general_research_agent:
  trace content-supervisor-flow  spans=9  tokens=11181
  llm calls: 4   tools: ['get_todays_date', 'advanced_research', 'advanced_research', 'advanced_research', 'advanced_research']
  distinct prompt versions stamped: 2
writing_agent:
  trace content-supervisor-flow  spans=5  tokens=3484
  llm calls: 3   tools: ['get_todays_date', 'basic_research']
  distinct prompt versions stamped: 2


Look at the last line of each block. More than one prompt version inside a single trace is the
supervisor pattern showing up in the data: the router's call and the subagent's calls are the same
run, done by different prompts.

The tool lists differ per subagent because their bindings differ. Token counts and the exact tool
order move on every run.

---

## Step 10 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — `response_format` and `tools` on the same request

**Broken on purpose.** This is the constraint that shapes the whole two-call design. Ask for both
and the request is rejected.

In [12]:
rendered = await hub.prompts.render(ROUTER_PROMPT, "production",
                                    {"question": "Anything about TSLA?"})
sub = await hub.prompts.render("finance-research-agent", "production", {"task": "TSLA"})

res = await rest.post("/gateway/chat/completions", json={
    "model": rendered.model,
    "messages": rendered.messages,
    "response_format": ROUTE_FORMAT,
    "tools": sub.tools,               # broken on purpose: both at once
})
print(f"HTTP {res.status_code}")
print(json.dumps(res.json())[:300])

HTTP 400
{"error": {"code": "VALIDATION_ERROR", "message": "response_format cannot be combined with tools, tool_choice, or tool_refs on the same request"}}


### Mistake 2 — you forget to thread the trace id

**Broken on purpose.** The quiet one. Leave the trace id out of Call B and both halves still work
perfectly — they just stop being one run, so nothing connects the routing decision to the work it
caused.

In [13]:
route, trace_a = await classify("What are the trends in sustainable packaging?")
rendered = await hub.prompts.render(SUBAGENT_PROMPTS[route], "production",
                                    {"task": "sustainable packaging trends, one sentence"})

split = await hub.gateway.run_tool_loop(
    rendered.model,
    rendered.messages,
    tools=[DECORATED[name] for name in TOOLS_BY_SUBAGENT[route]],
    sync=False,
    # Broken on purpose: no trace={"trace_id": trace_a}, so this opens its own trace.
)
await hub.gateway.flush()

print("Call A trace:", trace_a)
print("Call B trace:", split.trace_id)
print("same trace? ", trace_a == split.trace_id)
for label, tid in (("A", trace_a), ("B", split.trace_id)):
    detail = await hub.traces.get(tid)
    print(f"  trace {label}: name={detail.trace.name!r} spans={detail.trace.span_count}")

Call A trace: 355a98ef-0d50-47a2-840d-e3b22721a393
Call B trace: be4bf2e6-9ab9-4ba4-b8da-1e704a428bc5
same trace?  False
  trace A: name='content-supervisor-flow' spans=1
  trace B: name='runToolLoop' spans=4


### Mistake 3 — a router with no schema

**Broken on purpose.** Drop `response_format` and the router answers in prose. This is what the
text-protocol approach exists to cope with, and what the enum removes the need for.

In [14]:
rendered = await hub.prompts.render(ROUTER_PROMPT, "production",
                                    {"question": "Research Tesla stock news."})
res = await rest.post("/gateway/chat/completions", json={
    "model": rendered.model,
    "messages": rendered.messages,
    # Broken on purpose: no response_format at all.
})
content = res.json()["choices"][0]["message"]["content"]
print("the router said:\n ", content[:220], "\n")

try:
    print(json.loads(content)["route_to"])
except (json.JSONDecodeError, KeyError, TypeError) as err:
    print(f"{type(err).__name__}: {err}")
    print("SUBAGENT_PROMPTS[...] would raise KeyError on whatever you scraped out of that.")

the router said:
  I'll route this to the finance research agent since you're asking for Tesla stock news, which requires specialized financial data research.

**Routing to: finance_research_agent**

The finance research agent will search  

JSONDecodeError: Expecting value: line 1 column 1 (char 0)
SUBAGENT_PROMPTS[...] would raise KeyError on whatever you scraped out of that.


### Mistake 4 — trusting `response_format` on a passthrough connection

**Check.** The third trap from Step 1, tested for real — and the reason this one is a **Check**
rather than broken on purpose is itself the lesson. The gateway handles `response_format` natively
for a direct OpenAI connection and translates it into a forced tool call for a direct Anthropic
one. For an `openai_compatible` connection it forwards the field upstream and cannot make the
upstream honour it.

So this cell may well print clean JSON. That is the danger, not the reassurance: it worked on this
run, on this upstream, on this day, and nothing in the stack promises it will next time. A router
that parses JSON has to be on a connection where the schema is enforced.

If your team has no `openai_compatible` model, this cell says so and skips.

In [15]:
if PASSTHROUGH_MODEL is None:
    print("no openai_compatible model on this team - skipping")
else:
    rendered = await hub.prompts.render(ROUTER_PROMPT, "production",
                                        {"question": "Research Tesla stock news."})
    res = await rest.post("/gateway/chat/completions", json={
        "model": PASSTHROUGH_MODEL,          # broken on purpose: not a direct connection
        "messages": rendered.messages,
        "response_format": ROUTE_FORMAT,
    })
    print(f"model {PASSTHROUGH_MODEL!r} -> HTTP {res.status_code}")
    if res.status_code >= 400:
        print("  rejected outright:", json.dumps(res.json())[:200])
    else:
        content = res.json()["choices"][0]["message"]["content"]
        print("  content:", repr(content[:160]))
        try:
            print("  parsed route:", json.loads(content)["route_to"], "- honoured this time")
        except Exception as err:
            print(f"  {type(err).__name__}: the schema was ignored upstream")
    print("\nEither way: put a router on a direct connection, and do not rely on this.")

model 'mistral-small' -> HTTP 200
  content: '{ "route_to": "finance_research_agent" }'
  parsed route: finance_research_agent - honoured this time

Either way: put a router on a direct connection, and do not rely on this.


---

## Step 11 — Close the clients

**Your app.** Traces are reported in the background so they never slow your request down. Closing
the client flushes whatever is still queued. In a script `async with AcruxCore() as hub:` does it
for you; a notebook has no block to leave, so do it by hand.

The raw `rest` client from the preflight needs closing too.

In [16]:
await hub.gateway.aclose()
await rest.aclose()
print("flushed")

flushed


---

## What you built

A router and three specialists. One question in, one classification, one hand-off, one trace. No
framework, no state machine, and no text protocol — because the classification's answer was
constrained to three strings instead of being parsed out of prose.

### What of this actually ships

The two calls, plus the four implementations from Step 2:

```python
import json, os, httpx
from acruxcore import AcruxCore, acrux

SUBAGENT_PROMPTS = {
    "finance_research_agent": "finance-research-agent",
    "general_research_agent": "general-research-agent",
    "writing_agent": "writing-agent",
}
TOOLS_BY_SUBAGENT = {
    "finance_research_agent": ["finance_research", "basic_research", "get_todays_date"],
    "general_research_agent": ["advanced_research", "get_todays_date"],
    "writing_agent": ["basic_research", "get_todays_date"],
}
ROUTE_FORMAT = {"type": "json_schema", "json_schema": {
    "name": "route_decision", "strict": True,
    "schema": {"type": "object",
               "properties": {"route_to": {"type": "string",
                                           "enum": list(SUBAGENT_PROMPTS)}},
               "required": ["route_to"], "additionalProperties": False}}}


async def run_supervisor(question: str) -> str:
    async with AcruxCore() as hub:
        rest = httpx.AsyncClient(
            base_url=os.environ["ACRUXCORE_BASE_URL"],
            headers={"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}"},
            timeout=180)

        # Call A: classify. Raw httpx, because the trace id is only in a response header.
        router = await hub.prompts.render("content-supervisor", "production",
                                          {"question": question})
        res = await rest.post("/gateway/chat/completions",
                              headers={"x-trace-name": "content-supervisor-flow"},
                              json={"model": router.model, "messages": router.messages,
                                    "response_format": ROUTE_FORMAT,
                                    "prompt_version_id": router.version_id})
        res.raise_for_status()
        route = json.loads(res.json()["choices"][0]["message"]["content"])["route_to"]
        trace_id = res.headers["x-gateway-trace-id"]

        # Call B: dispatch, on the same trace.
        sub = await hub.prompts.render(SUBAGENT_PROMPTS[route], "production",
                                       {"task": question})
        result = await hub.gateway.run_tool_loop(
            sub.model, sub.messages,
            tools=[DECORATED[name] for name in TOOLS_BY_SUBAGENT[route]],
            sync=False, prompt_version_id=sub.version_id,
            trace={"trace_id": trace_id, "name": "content-supervisor-flow"})

        await rest.aclose()
        return result.content
```

Everything else was scaffolding:

- `find_prompt_by_name` and `create_prompt_if_missing` exist so this notebook can be re-run. They
  are notebook helpers, not SDK calls.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, `resolve`, the trace walk — proves a step worked. None of
  it belongs in a request path.
- Step 10 is all deliberately broken, and it leaves two extra traces behind.

### Adding a fourth specialist

Nothing in the shipping code above names a specialist. To add one: commit a prompt, bind its
tools, and add one entry to `SUBAGENT_PROMPTS` and one to `TOOLS_BY_SUBAGENT`. The enum in
`ROUTE_FORMAT` is built from `SUBAGENT_PROMPTS`, so the router learns about it automatically —
but its **description** has to go in the router's system message, or the model has a name it knows
nothing about.

### Going multi-hop

This is the one-hop case: classify once, dispatch once. To chain — research first, then write from
that research — feed the subagent's output back into a second `classify` call, on the same trace
id, and stop when the router says no further work is needed. That needs a loop bound, because a
router asked "is this done" can say no forever.

### What this notebook left in your team

- four tools at v1, all `client` executors
- four prompts at v1: the router plus three subagents
- seven bindings, none on the router
- a handful of traces: three good runs, plus the extras Step 7 and Step 10 leave behind

### Where to go next

- [Build a Medical-Information QA Agent](https://docs.acruxcore.com/docs/tutorials/build-a-medical-information-qa-agent)
  — `response_format` in depth, including what a schema cannot promise.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.